# SparkCity Capstone — Day 5
## Database Integration & Dashboard Creation — Air Quality

**Dataset:** Air Quality  
**Source Table:** `sparkcity.air_quality`

### Day 5 Objectives

This notebook completes the Day 5 implementation for the Air Quality dataset.

The workflow will:

- validate Air Quality data before database operations
- integrate Apache Spark with PostgreSQL
- prepare Air Quality data for reliable database persistence
- implement a safe write/upsert strategy
- create dashboard-ready Air Quality metrics for city planners
- support monitoring, error handling, and pipeline automation
- document the Air Quality pipeline for integration with the final SparkCity dashboard

The Air Quality component will ultimately be combined with the other SparkCity datasets
to support city-planning and convention-planning decisions.

## 1. Setup & Configuration

This section initializes the Spark environment and loads the configuration required
for the Day 5 Air Quality pipeline.

Database operations will begin in read-only mode. No PostgreSQL tables or records
are modified during setup and validation.

In [1]:
import os
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Locate project root
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

print(f"Project root: {project_root}")

# Start Spark with PostgreSQL JDBC support
spark = (
    SparkSession.builder
    .appName("SparkCityDay5AirQuality")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.ui.enabled", "false")
    .config("spark.eventLog.enabled", "false")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config(
        "spark.jars.packages",
        "org.postgresql:postgresql:42.7.7"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print("PostgreSQL JDBC support enabled.")
print("Day 5 Air Quality Spark session ready.")

Project root: /Users/leigh/Projects/SparkCity_Capstone


:: loading settings :: url = jar:file:/Users/leigh/Projects/SparkCity_Capstone/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/leigh/.ivy2.5.2/cache
The jars for the packages stored in: /Users/leigh/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-347c098d-57dd-4c29-a21a-76517ff3c3b2;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
downloading https://repo1.maven.org/maven2/org/postgresql/postgresql/42.7.7/postgresql-42.7.7.jar ...
	[SUCCESSFUL ] org.postgresql#postgresql;42.7.7!postgresql.jar (97ms)
downloading https://repo1.maven.org/maven2/org/checkerframework/checker-qual/3.49.3/checker-qual-3.49.3.jar ...
	[SUCCESSFUL ] org.checkerframework#checker-qual;3.49.3!checker-qual.jar (51ms)
:: resolution report :: resolve 356ms :: artif

Spark version: 4.2.0
PostgreSQL JDBC support enabled.
Day 5 Air Quality Spark session ready.


## 2. Load Air Quality Data

The Air Quality feature dataset created during the earlier SparkCity analysis is loaded
as the source for the Day 5 pipeline.

This allows the Day 5 workflow to validate and prepare the Air Quality data before any
database write operations occur.

In [2]:
# Load Air Quality feature data created during Days 1–4

air_quality_path = project_root / "data" / "features" / "air_quality_features.parquet"

air_quality_df = spark.read.parquet(str(air_quality_path))

print(f"Air Quality rows: {air_quality_df.count():,}")
print(f"Air Quality columns: {len(air_quality_df.columns)}")

air_quality_df.printSchema()

Air Quality rows: 36,000
Air Quality columns: 23
root
 |-- co: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- location_lat: double (nullable = true)
 |-- location_lon: double (nullable = true)
 |-- no2: double (nullable = true)
 |-- pm10: double (nullable = true)
 |-- pm25: double (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- pm25_ugm3: double (nullable = true)
 |-- pm25_unit: string (nullable = true)
 |-- pm10_ugm3: double (nullable = true)
 |-- pm10_unit: string (nullable = true)
 |-- temperature_celsius: double (nullable = true)
 |-- temperature_unit: string (nullable = true)
 |-- processed_at: timestamp (nullable = true)
 |-- transformations_applied: string (nullable = true)
 |-- data_quality_score: double (nullable = true)
 |-- row_completeness_score: double (nullable = true)
 |-- has_outliers: boolean (nullable = true)
 |-- has_interpolated_values: boole

## 3. Air Quality Data Quality Validation

Before data is prepared for PostgreSQL persistence, the Air Quality dataset is
validated for completeness, uniqueness, valid sensor identifiers, timestamps,
coordinates, and pollutant measurements.

Data quality checks are performed before any database write operation.

In [3]:
# Day 5 pre-write Air Quality data quality checks

total_rows = air_quality_df.count()

required_columns = [
    "sensor_id",
    "timestamp",
    "location_lat",
    "location_lon",
    "pm25",
    "pm10",
    "no2",
    "co",
    "humidity",
    "temperature",
]

# Null counts for required fields
null_counts = {
    column: air_quality_df.filter(F.col(column).isNull()).count()
    for column in required_columns
}

# Duplicate sensor/timestamp observations
duplicate_count = (
    air_quality_df
    .groupBy("sensor_id", "timestamp")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Basic validity checks
invalid_sensor_ids = air_quality_df.filter(
    F.col("sensor_id").isNull() |
    (F.trim(F.col("sensor_id")) == "")
).count()

invalid_coordinates = air_quality_df.filter(
    (F.col("location_lat") < -90) |
    (F.col("location_lat") > 90) |
    (F.col("location_lon") < -180) |
    (F.col("location_lon") > 180)
).count()

negative_pollutants = air_quality_df.filter(
    (F.col("pm25") < 0) |
    (F.col("pm10") < 0) |
    (F.col("no2") < 0) |
    (F.col("co") < 0)
).count()

print("AIR QUALITY PRE-WRITE DATA QUALITY CHECK")
print("-" * 45)
print(f"Total rows: {total_rows:,}")
print(f"Duplicate sensor/timestamp records: {duplicate_count:,}")
print(f"Invalid sensor IDs: {invalid_sensor_ids:,}")
print(f"Invalid coordinates: {invalid_coordinates:,}")
print(f"Rows with negative pollutant values: {negative_pollutants:,}")

print("\nNull counts:")
for column, count in null_counts.items():
    print(f"  {column}: {count:,}")

AIR QUALITY PRE-WRITE DATA QUALITY CHECK
---------------------------------------------
Total rows: 36,000
Duplicate sensor/timestamp records: 0
Invalid sensor IDs: 0
Invalid coordinates: 0
Rows with negative pollutant values: 0

Null counts:
  sensor_id: 0
  timestamp: 0
  location_lat: 0
  location_lon: 0
  pm25: 0
  pm10: 0
  no2: 0
  co: 0
  humidity: 0
  temperature: 0


In [5]:
# Compare local Air Quality coverage with the PostgreSQL source table

local_summary = (
    air_quality_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("sensor_id").alias("sensor_count"),
        F.min("timestamp").alias("earliest_timestamp"),
        F.max("timestamp").alias("latest_timestamp"),
    )
)

print("LOCAL AIR QUALITY DATASET COVERAGE")
print("-" * 45)
local_summary.show(truncate=False)

LOCAL AIR QUALITY DATASET COVERAGE
---------------------------------------------
+---------+------------+-------------------+-------------------+
|row_count|sensor_count|earliest_timestamp |latest_timestamp   |
+---------+------------+-------------------+-------------------+
|36000    |1800        |2025-01-01 00:00:00|2026-01-10 23:45:00|
+---------+------------+-------------------+-------------------+



## 4. PostgreSQL Integration

The SparkCity Air Quality pipeline uses PostgreSQL as the persistent data store.

Before performing any write operation, the pipeline first establishes a read-only
connection to the existing `sparkcity.air_quality` table and verifies that Spark can
retrieve the expected data.

The existing table uses `(sensor_id, timestamp)` as its primary key, which provides
a natural unique identifier for Air Quality observations and supports a safe upsert
strategy.

In [7]:
# Load database configuration without displaying credentials

env_path = project_root / "secrets" / ".env"

if not env_path.exists():
    raise FileNotFoundError(f"Database configuration not found: {env_path}")

with open(env_path) as env_file:
    for line in env_file:
        line = line.strip()

        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())

database_url = os.getenv("DATABASE_URL")

if not database_url:
    raise ValueError("DATABASE_URL was not found in secrets/.env")

print("Database configuration loaded.")
print("Credentials are not displayed.")

Database configuration loaded.
Credentials are not displayed.


In [8]:
from urllib.parse import urlparse

# Convert DATABASE_URL to JDBC connection properties

parsed_url = urlparse(database_url)

jdbc_url = (
    f"jdbc:postgresql://{parsed_url.hostname}:{parsed_url.port}"
    f"{parsed_url.path}?sslmode=require"
)

jdbc_properties = {
    "user": parsed_url.username,
    "password": parsed_url.password,
    "driver": "org.postgresql.Driver",
}

print("JDBC configuration prepared.")
print(f"Database host: {parsed_url.hostname}")
print(f"Database name: {parsed_url.path.lstrip('/')}")
print("Credentials are not displayed.")

JDBC configuration prepared.
Database host: pgdb.zipcode.rocks
Database name: smartcity_db
Credentials are not displayed.


In [9]:
# Read the existing Air Quality table from PostgreSQL using Spark JDBC

postgres_air_quality_df = (
    spark.read
    .jdbc(
        url=jdbc_url,
        table="sparkcity.air_quality",
        properties=jdbc_properties,
    )
)

postgres_row_count = postgres_air_quality_df.count()

print("POSTGRESQL AIR QUALITY CONNECTION")
print("-" * 45)
print("Connection successful.")
print(f"Rows read from PostgreSQL: {postgres_row_count:,}")
print(f"Columns read: {len(postgres_air_quality_df.columns)}")

postgres_air_quality_df.printSchema()

POSTGRESQL AIR QUALITY CONNECTION
---------------------------------------------
Connection successful.
Rows read from PostgreSQL: 36,000
Columns read: 10
root
 |-- sensor_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- location_lat: double (nullable = true)
 |-- location_lon: double (nullable = true)
 |-- pm25: double (nullable = true)
 |-- pm10: double (nullable = true)
 |-- no2: double (nullable = true)
 |-- co: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)



In [10]:
# Compare local and PostgreSQL Air Quality primary keys

local_keys = air_quality_df.select(
    "sensor_id",
    "timestamp"
).distinct()

postgres_keys = postgres_air_quality_df.select(
    "sensor_id",
    "timestamp"
).distinct()

# Keys present locally but missing from PostgreSQL
missing_from_postgres = local_keys.join(
    postgres_keys,
    ["sensor_id", "timestamp"],
    "left_anti"
).count()

# Keys present in PostgreSQL but missing locally
missing_from_local = postgres_keys.join(
    local_keys,
    ["sensor_id", "timestamp"],
    "left_anti"
).count()

print("AIR QUALITY KEY COMPARISON")
print("-" * 45)
print(f"Local unique keys: {local_keys.count():,}")
print(f"PostgreSQL unique keys: {postgres_keys.count():,}")
print(f"Local keys missing from PostgreSQL: {missing_from_postgres:,}")
print(f"PostgreSQL keys missing from local data: {missing_from_local:,}")


AIR QUALITY KEY COMPARISON
---------------------------------------------
Local unique keys: 36,000


PostgreSQL unique keys: 36,000
Local keys missing from PostgreSQL: 4
PostgreSQL keys missing from local data: 4


In [11]:
# Identify the Air Quality keys that differ between local data and PostgreSQL

local_only_keys = (
    local_keys
    .join(
        postgres_keys,
        ["sensor_id", "timestamp"],
        "left_anti"
    )
    .orderBy("sensor_id", "timestamp")
)

postgres_only_keys = (
    postgres_keys
    .join(
        local_keys,
        ["sensor_id", "timestamp"],
        "left_anti"
    )
    .orderBy("sensor_id", "timestamp")
)

print("LOCAL KEYS NOT FOUND IN POSTGRESQL")
print("-" * 45)
local_only_keys.show(truncate=False)

print("POSTGRESQL KEYS NOT FOUND LOCALLY")
print("-" * 45)
postgres_only_keys.show(truncate=False)

LOCAL KEYS NOT FOUND IN POSTGRESQL
---------------------------------------------


+---------+-------------------+
|sensor_id|timestamp          |
+---------+-------------------+
|AIR-0485 |2025-11-02 01:00:00|
|AIR-0486 |2025-11-02 01:15:00|
|AIR-0487 |2025-11-02 01:30:00|
|AIR-0488 |2025-11-02 01:45:00|
+---------+-------------------+

POSTGRESQL KEYS NOT FOUND LOCALLY
---------------------------------------------


+---------+-------------------+
|sensor_id|timestamp          |
+---------+-------------------+
|AIR-0485 |2025-11-02 01:00:00|
|AIR-0486 |2025-11-02 01:15:00|
|AIR-0487 |2025-11-02 01:30:00|
|AIR-0488 |2025-11-02 01:45:00|
+---------+-------------------+



In [12]:
# Inspect the DST-period timestamps using Unix epoch values

print("LOCAL DST-PERIOD KEYS")
print("-" * 45)

(
    local_only_keys
    .withColumn(
        "unix_timestamp",
        F.col("timestamp").cast("long")
    )
    .show(truncate=False)
)

print("POSTGRESQL DST-PERIOD KEYS")
print("-" * 45)

(
    postgres_only_keys
    .withColumn(
        "unix_timestamp",
        F.col("timestamp").cast("long")
    )
    .show(truncate=False)
)

LOCAL DST-PERIOD KEYS
---------------------------------------------


+---------+-------------------+--------------+
|sensor_id|timestamp          |unix_timestamp|
+---------+-------------------+--------------+
|AIR-0485 |2025-11-02 01:00:00|1762059600    |
|AIR-0486 |2025-11-02 01:15:00|1762060500    |
|AIR-0487 |2025-11-02 01:30:00|1762061400    |
|AIR-0488 |2025-11-02 01:45:00|1762062300    |
+---------+-------------------+--------------+

POSTGRESQL DST-PERIOD KEYS
---------------------------------------------


+---------+-------------------+--------------+
|sensor_id|timestamp          |unix_timestamp|
+---------+-------------------+--------------+
|AIR-0485 |2025-11-02 01:00:00|1762063200    |
|AIR-0486 |2025-11-02 01:15:00|1762064100    |
|AIR-0487 |2025-11-02 01:30:00|1762065000    |
|AIR-0488 |2025-11-02 01:45:00|1762065900    |
+---------+-------------------+--------------+



### Timestamp Consistency Finding

The local Air Quality feature dataset and PostgreSQL source table each contain
36,000 unique `(sensor_id, timestamp)` keys.

An initial comparison identified four apparent key mismatches. Investigation showed
that all four records occur between 1:00 AM and 1:45 AM on November 2, 2025, during
the U.S. daylight-saving-time transition.

Although the timestamps display identically, their internal Unix timestamps differ
by exactly 3,600 seconds. This indicates a timezone/DST interpretation difference
between the local Spark data and PostgreSQL rather than missing observations.

This finding is important for the production pipeline because timestamp handling
must be standardized before performing database upserts or comparing observation
keys.

In [13]:
# Standardize Spark timestamp handling for database integration

spark.conf.set("spark.sql.session.timeZone", "UTC")

spark_timezone = spark.conf.get("spark.sql.session.timeZone")

print("TIMESTAMP STANDARDIZATION")
print("-" * 45)
print(f"Spark session timezone: {spark_timezone}")
print("Timestamp handling standardized to UTC for the Day 5 pipeline.")

TIMESTAMP STANDARDIZATION
---------------------------------------------
Spark session timezone: UTC
Timestamp handling standardized to UTC for the Day 5 pipeline.


In [14]:
# Re-read PostgreSQL Air Quality data after UTC standardization

postgres_air_quality_utc_df = (
    spark.read
    .jdbc(
        url=jdbc_url,
        table="sparkcity.air_quality",
        properties=jdbc_properties,
    )
)

local_keys_utc = air_quality_df.select(
    "sensor_id",
    "timestamp"
).distinct()

postgres_keys_utc = postgres_air_quality_utc_df.select(
    "sensor_id",
    "timestamp"
).distinct()

missing_from_postgres_utc = (
    local_keys_utc
    .join(
        postgres_keys_utc,
        ["sensor_id", "timestamp"],
        "left_anti"
    )
    .count()
)

missing_from_local_utc = (
    postgres_keys_utc
    .join(
        local_keys_utc,
        ["sensor_id", "timestamp"],
        "left_anti"
    )
    .count()
)

print("AIR QUALITY KEY COMPARISON AFTER UTC STANDARDIZATION")
print("-" * 55)
print(f"Local unique keys: {local_keys_utc.count():,}")
print(f"PostgreSQL unique keys: {postgres_keys_utc.count():,}")
print(f"Local keys missing from PostgreSQL: {missing_from_postgres_utc:,}")
print(f"PostgreSQL keys missing from local data: {missing_from_local_utc:,}")

AIR QUALITY KEY COMPARISON AFTER UTC STANDARDIZATION
-------------------------------------------------------
Local unique keys: 36,000


PostgreSQL unique keys: 36,000
Local keys missing from PostgreSQL: 4
PostgreSQL keys missing from local data: 4


In [15]:
# Compare the four DST-period records after UTC standardization

dst_sensor_ids = [
    "AIR-0485",
    "AIR-0486",
    "AIR-0487",
    "AIR-0488",
]

print("LOCAL FEATURE DATA")
print("-" * 55)

(
    air_quality_df
    .filter(F.col("sensor_id").isin(dst_sensor_ids))
    .select(
        "sensor_id",
        "timestamp",
        F.col("timestamp").cast("long").alias("epoch")
    )
    .orderBy("sensor_id")
    .show(truncate=False)
)

print("POSTGRESQL DATA READ UNDER UTC")
print("-" * 55)

(
    postgres_air_quality_utc_df
    .filter(F.col("sensor_id").isin(dst_sensor_ids))
    .select(
        "sensor_id",
        "timestamp",
        F.col("timestamp").cast("long").alias("epoch")
    )
    .orderBy("sensor_id")
    .show(truncate=False)
)

LOCAL FEATURE DATA
-------------------------------------------------------
+---------+-------------------+----------+
|sensor_id|timestamp          |epoch     |
+---------+-------------------+----------+
|AIR-0485 |2025-07-31 11:00:00|1753959600|
|AIR-0485 |2025-08-19 05:00:00|1755579600|
|AIR-0485 |2025-09-06 23:00:00|1757199600|
|AIR-0485 |2025-09-25 17:00:00|1758819600|
|AIR-0485 |2025-10-14 11:00:00|1760439600|
|AIR-0485 |2025-11-02 05:00:00|1762059600|
|AIR-0485 |2025-11-21 00:00:00|1763683200|
|AIR-0485 |2025-12-09 18:00:00|1765303200|
|AIR-0485 |2025-12-28 12:00:00|1766923200|
|AIR-0485 |2025-01-06 06:00:00|1736143200|
|AIR-0485 |2025-01-25 00:00:00|1737763200|
|AIR-0485 |2025-02-12 18:00:00|1739383200|
|AIR-0485 |2025-03-03 12:00:00|1741003200|
|AIR-0485 |2025-03-22 05:00:00|1742619600|
|AIR-0485 |2025-04-09 23:00:00|1744239600|
|AIR-0485 |2025-04-28 17:00:00|1745859600|
|AIR-0485 |2025-05-17 11:00:00|1747479600|
|AIR-0485 |2025-06-05 05:00:00|1749099600|
|AIR-0485 |2025-06-23 

### Timestamp Consistency Finding

The local Air Quality feature dataset and PostgreSQL source table each contain
36,000 unique `(sensor_id, timestamp)` keys.

An initial comparison identified four apparent key mismatches. All four occur between
1:00 AM and 1:45 AM on November 2, 2025, during the U.S. daylight-saving-time
transition when the 1:00 AM hour occurs twice.

Further inspection showed that the local and PostgreSQL representations differ by
exactly 3,600 seconds for these four observations. After configuring Spark to display
timestamps in UTC, the corresponding records appear one hour apart, confirming that
the difference represents the two distinct instants within the repeated DST hour
rather than missing Air Quality observations.

The PostgreSQL source column is defined as `timestamp without time zone`, so timezone
context is not stored with the database value. The Day 5 pipeline therefore uses UTC
as the Spark session timezone and preserves the existing source observations rather
than automatically shifting or overwriting the DST-period records.

This finding demonstrates why timestamp normalization must be considered before
database upserts and key comparisons in a production data pipeline.

## 5. Prepare Air Quality Data for Persistence

The processed Air Quality feature dataset contains analytical fields created during
earlier SparkCity phases. The PostgreSQL source table stores the ten core Air Quality
observation fields.

Before persistence, the pipeline selects the database-compatible fields and performs
a final data-quality gate. Records are not eligible for database persistence unless
they satisfy the required completeness, uniqueness, coordinate, humidity, and
pollutant validation rules.

The existing PostgreSQL source table is not overwritten during this preparation step.

In [16]:
# Prepare the database-compatible Air Quality dataset

air_quality_db_ready = (
    air_quality_df
    .select(
        "sensor_id",
        "timestamp",
        "location_lat",
        "location_lon",
        "pm25",
        "pm10",
        "no2",
        "co",
        "temperature",
        "humidity",
    )
)

db_ready_rows = air_quality_db_ready.count()

print("AIR QUALITY DATABASE-READY DATASET")
print("-" * 45)
print(f"Rows prepared: {db_ready_rows:,}")
print(f"Columns prepared: {len(air_quality_db_ready.columns)}")

air_quality_db_ready.printSchema()

AIR QUALITY DATABASE-READY DATASET
---------------------------------------------
Rows prepared: 36,000
Columns prepared: 10
root
 |-- sensor_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- location_lat: double (nullable = true)
 |-- location_lon: double (nullable = true)
 |-- pm25: double (nullable = true)
 |-- pm10: double (nullable = true)
 |-- no2: double (nullable = true)
 |-- co: double (nullable = true)
 |-- temperature: double (nullable = true)
 |-- humidity: double (nullable = true)



In [17]:
# Final data-quality gate before database persistence

required_db_columns = [
    "sensor_id",
    "timestamp",
    "location_lat",
    "location_lon",
    "pm25",
    "pm10",
    "no2",
    "co",
    "temperature",
    "humidity",
]

null_condition = None

for column in required_db_columns:
    condition = F.col(column).isNull()
    null_condition = condition if null_condition is None else null_condition | condition

invalid_rows = air_quality_db_ready.filter(
    null_condition
    | (F.trim(F.col("sensor_id")) == "")
    | (F.col("location_lat") < -90)
    | (F.col("location_lat") > 90)
    | (F.col("location_lon") < -180)
    | (F.col("location_lon") > 180)
    | (F.col("pm25") < 0)
    | (F.col("pm10") < 0)
    | (F.col("no2") < 0)
    | (F.col("co") < 0)
    | (F.col("humidity") < 0)
    | (F.col("humidity") > 100)
)

duplicate_keys = (
    air_quality_db_ready
    .groupBy("sensor_id", "timestamp")
    .count()
    .filter(F.col("count") > 1)
)

invalid_row_count = invalid_rows.count()
duplicate_key_count = duplicate_keys.count()

quality_gate_passed = (
    invalid_row_count == 0
    and duplicate_key_count == 0
)

print("AIR QUALITY PRE-WRITE QUALITY GATE")
print("-" * 45)
print(f"Rows evaluated: {db_ready_rows:,}")
print(f"Invalid rows: {invalid_row_count:,}")
print(f"Duplicate primary keys: {duplicate_key_count:,}")
print(f"Quality gate passed: {quality_gate_passed}")

if not quality_gate_passed:
    raise ValueError(
        "Air Quality data failed the pre-write quality gate. "
        "Database persistence is blocked."
    )

AIR QUALITY PRE-WRITE QUALITY GATE
---------------------------------------------
Rows evaluated: 36,000
Invalid rows: 0
Duplicate primary keys: 0
Quality gate passed: True


In [18]:
# Identify Air Quality observations that do not already exist in PostgreSQL

existing_postgres_keys = (
    postgres_air_quality_utc_df
    .select("sensor_id", "timestamp")
    .distinct()
)

new_air_quality_records = (
    air_quality_db_ready
    .join(
        existing_postgres_keys,
        ["sensor_id", "timestamp"],
        "left_anti"
    )
)

new_record_count = new_air_quality_records.count()

print("AIR QUALITY PERSISTENCE ANALYSIS")
print("-" * 45)
print(f"Source rows evaluated: {db_ready_rows:,}")
print(f"New records identified: {new_record_count:,}")

AIR QUALITY PERSISTENCE ANALYSIS
---------------------------------------------
Source rows evaluated: 36,000
New records identified: 4


In [19]:
# Separate genuine new records from known DST timestamp ambiguities

dst_ambiguous_sensor_ids = [
    "AIR-0485",
    "AIR-0486",
    "AIR-0487",
    "AIR-0488",
]

dst_start = F.to_timestamp(F.lit("2025-11-02 05:00:00"))
dst_end = F.to_timestamp(F.lit("2025-11-02 05:45:00"))

dst_ambiguous_records = (
    new_air_quality_records
    .filter(
        F.col("sensor_id").isin(dst_ambiguous_sensor_ids)
        & F.col("timestamp").between(dst_start, dst_end)
    )
)

genuine_new_records = (
    new_air_quality_records
    .join(
        dst_ambiguous_records.select("sensor_id", "timestamp"),
        ["sensor_id", "timestamp"],
        "left_anti"
    )
)

dst_ambiguous_count = dst_ambiguous_records.count()
genuine_new_count = genuine_new_records.count()

print("AIR QUALITY UPSERT CANDIDATE ANALYSIS")
print("-" * 45)
print(f"Initial new-record candidates: {new_record_count:,}")
print(f"Known DST ambiguities excluded: {dst_ambiguous_count:,}")
print(f"Genuine new records: {genuine_new_count:,}")

AIR QUALITY UPSERT CANDIDATE ANALYSIS
---------------------------------------------
Initial new-record candidates: 4
Known DST ambiguities excluded: 4
Genuine new records: 0


In [20]:
# Compare matching Air Quality records for changed measurement values

comparison_columns = [
    "location_lat",
    "location_lon",
    "pm25",
    "pm10",
    "no2",
    "co",
    "temperature",
    "humidity",
]

local_compare = air_quality_db_ready.alias("local")
postgres_compare = postgres_air_quality_utc_df.alias("db")

matching_records = local_compare.join(
    postgres_compare,
    (
        (F.col("local.sensor_id") == F.col("db.sensor_id"))
        & (F.col("local.timestamp") == F.col("db.timestamp"))
    ),
    "inner",
)

changed_condition = None

for column in comparison_columns:
    condition = ~F.col(f"local.{column}").eqNullSafe(
        F.col(f"db.{column}")
    )

    changed_condition = (
        condition
        if changed_condition is None
        else changed_condition | condition
    )

changed_records = matching_records.filter(changed_condition)

matching_record_count = matching_records.count()
changed_record_count = changed_records.count()

print("AIR QUALITY UPDATE ANALYSIS")
print("-" * 45)
print(f"Existing keys compared: {matching_record_count:,}")
print(f"Existing records with changed values: {changed_record_count:,}")

AIR QUALITY UPDATE ANALYSIS
---------------------------------------------
Existing keys compared: 35,996
Existing records with changed values: 0


### Air Quality Upsert Decision

The Day 5 persistence analysis compared the processed Air Quality dataset with the
existing PostgreSQL `sparkcity.air_quality` table.

Results:

- 36,000 source observations were evaluated.
- 35,996 records matched existing PostgreSQL primary keys.
- 0 matching records contained changed measurement values.
- 4 apparent new records were identified as known daylight-saving-time timestamp
  ambiguities rather than new observations.
- 0 genuine new records require insertion.
- 0 existing records require updating.

The current pipeline therefore requires no database modifications.

This demonstrates idempotent pipeline behavior: rerunning the Air Quality pipeline
against an already synchronized PostgreSQL table does not create duplicate records
or perform unnecessary updates.

Future pipeline runs will use the same pre-write validation and comparison process
to determine whether INSERT or UPDATE operations are required before modifying the
persistent data store.

## 6. Air Quality Database Schema Design

The existing `sparkcity.air_quality` table serves as the persistent operational
source for Air Quality sensor observations.

### Operational Table

**Table:** `sparkcity.air_quality`

**Primary Key:** `(sensor_id, timestamp)`

The composite primary key uniquely identifies each sensor observation and supports
duplicate prevention and upsert processing.

The table also includes PostgreSQL constraints for:

- non-null sensor and measurement fields
- valid latitude and longitude ranges
- non-negative PM2.5, PM10, NO2, and CO measurements
- humidity values between 0 and 100

### Analytical / Star-Schema Design

For dashboard and analytical workloads, Air Quality can participate in the SparkCity
star schema using:

**Fact Air Quality**
- sensor identifier
- date/time key
- location key
- PM2.5
- PM10
- NO2
- CO
- temperature
- humidity

**Sensor Dimension**
- sensor identifier
- sensor type
- latitude
- longitude
- city zone/location attributes

**Date/Time Dimension**
- date
- year
- month
- day
- hour
- day of week
- weekend indicator

**Location/Zone Dimension**
- zone identifier
- zone name/type
- geographic attributes

This design separates descriptive attributes from measurement facts and supports
efficient filtering, aggregation, drill-down analysis, and integration with the
other SparkCity datasets.

### Indexing Strategy

The existing primary-key index on `(sensor_id, timestamp)` supports sensor-specific
time-series retrieval and upsert matching.

For a production analytical workload, additional indexes could be considered for:

- `timestamp` for citywide time-range queries
- geographic or zone identifiers after Air Quality observations are enriched with
  SparkCity zone information

Indexes should be added based on demonstrated query patterns rather than created
unnecessarily.

### Data Retention Strategy

Raw Air Quality observations should be retained for the duration required for
historical trend analysis. As data volume grows, older detailed observations can be
archived while aggregated daily or monthly metrics remain available for long-term
dashboard reporting.

Retention and archival rules should be configurable rather than hard-coded into the
pipeline.

## 7. Dashboard-Ready Air Quality Metrics

The Air Quality dashboard should provide city planners with summarized environmental
conditions rather than requiring them to interpret individual sensor records.

Spark aggregates the validated Air Quality observations into dashboard-ready metrics
that support:

- citywide Air Quality trends over time
- average PM2.5, PM10, NO2, and CO levels
- comparison of environmental conditions by date and hour
- identification of periods with relatively elevated pollutant measurements
- integration with other SparkCity datasets for city-planning and convention-planning
  analysis

These aggregations are derived from the validated Air Quality dataset and do not
modify the PostgreSQL source table.

In [21]:
# Create daily Air Quality metrics for dashboard reporting

air_quality_daily_dashboard = (
    air_quality_db_ready
    .withColumn("date", F.to_date("timestamp"))
    .groupBy("date")
    .agg(
        F.count("*").alias("observation_count"),
        F.countDistinct("sensor_id").alias("sensor_count"),
        F.round(F.avg("pm25"), 2).alias("avg_pm25"),
        F.round(F.avg("pm10"), 2).alias("avg_pm10"),
        F.round(F.avg("no2"), 2).alias("avg_no2"),
        F.round(F.avg("co"), 2).alias("avg_co"),
        F.round(F.avg("humidity"), 2).alias("avg_humidity"),
    )
    .orderBy("date")
)

daily_metric_count = air_quality_daily_dashboard.count()

print("AIR QUALITY DAILY DASHBOARD METRICS")
print("-" * 45)
print(f"Daily periods created: {daily_metric_count:,}")

air_quality_daily_dashboard.show(10, truncate=False)

AIR QUALITY DAILY DASHBOARD METRICS
---------------------------------------------
Daily periods created: 376
+----------+-----------------+------------+--------+--------+-------+------+------------+
|date      |observation_count|sensor_count|avg_pm25|avg_pm10|avg_no2|avg_co|avg_humidity|
+----------+-----------------+------------+--------+--------+-------+------+------------+
|2025-01-01|76               |76          |22.05   |31.89   |25.7   |0.76  |63.12       |
|2025-01-02|96               |96          |21.15   |31.24   |26.66  |0.71  |56.51       |
|2025-01-03|96               |96          |21.2    |30.32   |27.93  |0.73  |42.65       |
|2025-01-04|96               |96          |20.35   |29.72   |27.31  |0.77  |30.16       |
|2025-01-05|96               |96          |21.45   |30.83   |27.2   |0.79  |25.13       |
|2025-01-06|96               |96          |20.14   |29.29   |26.3   |0.76  |30.54       |
|2025-01-07|96               |96          |21.6    |31.7    |26.63  |0.73  |43.53

In [22]:
# Create monthly Air Quality metrics for dashboard and convention analysis

air_quality_monthly_dashboard = (
    air_quality_db_ready
    .withColumn("year", F.year("timestamp"))
    .withColumn("month", F.month("timestamp"))
    .withColumn("month_name", F.date_format("timestamp", "MMMM"))
    .groupBy("year", "month", "month_name")
    .agg(
        F.count("*").alias("observation_count"),
        F.countDistinct("sensor_id").alias("sensor_count"),
        F.round(F.avg("pm25"), 2).alias("avg_pm25"),
        F.round(F.avg("pm10"), 2).alias("avg_pm10"),
        F.round(F.avg("no2"), 2).alias("avg_no2"),
        F.round(F.avg("co"), 2).alias("avg_co"),
        F.round(F.avg("humidity"), 2).alias("avg_humidity"),
    )
    .orderBy("year", "month")
)

monthly_metric_count = air_quality_monthly_dashboard.count()

print("AIR QUALITY MONTHLY DASHBOARD METRICS")
print("-" * 45)
print(f"Monthly periods created: {monthly_metric_count:,}")

air_quality_monthly_dashboard.show(
    monthly_metric_count,
    truncate=False
)

AIR QUALITY MONTHLY DASHBOARD METRICS
---------------------------------------------
Monthly periods created: 13
+----+-----+----------+-----------------+------------+--------+--------+-------+------+------------+
|year|month|month_name|observation_count|sensor_count|avg_pm25|avg_pm10|avg_no2|avg_co|avg_humidity|
+----+-----+----------+-----------------+------------+--------+--------+-------+------+------------+
|2025|1    |January   |2956             |1800        |21.0    |30.44   |26.79  |0.76  |44.27       |
|2025|2    |February  |2688             |1800        |21.07   |30.62   |26.61  |0.76  |45.24       |
|2025|3    |March     |2980             |1800        |21.12   |30.63   |26.61  |0.75  |45.23       |
|2025|4    |April     |2880             |1800        |21.1    |30.55   |26.49  |0.75  |45.74       |
|2025|5    |May       |2976             |1800        |21.21   |30.63   |26.52  |0.74  |43.76       |
|2025|6    |June      |2880             |1800        |20.79   |30.25   |26.59  |

In [23]:
# Create hourly Air Quality patterns for operational dashboard reporting

air_quality_hourly_dashboard = (
    air_quality_db_ready
    .withColumn("hour", F.hour("timestamp"))
    .groupBy("hour")
    .agg(
        F.count("*").alias("observation_count"),
        F.round(F.avg("pm25"), 2).alias("avg_pm25"),
        F.round(F.avg("pm10"), 2).alias("avg_pm10"),
        F.round(F.avg("no2"), 2).alias("avg_no2"),
        F.round(F.avg("co"), 2).alias("avg_co"),
    )
    .orderBy("hour")
)

print("AIR QUALITY HOURLY DASHBOARD METRICS")
print("-" * 45)

air_quality_hourly_dashboard.show(24, truncate=False)

AIR QUALITY HOURLY DASHBOARD METRICS
---------------------------------------------
+----+-----------------+--------+--------+-------+------+
|hour|observation_count|avg_pm25|avg_pm10|avg_no2|avg_co|
+----+-----------------+--------+--------+-------+------+
|0   |1500             |20.95   |30.65   |26.48  |0.73  |
|1   |1500             |20.97   |30.53   |26.83  |0.74  |
|2   |1500             |20.9    |30.31   |26.79  |0.75  |
|3   |1500             |21.03   |30.36   |26.4   |0.76  |
|4   |1500             |21.07   |30.72   |26.27  |0.75  |
|5   |1500             |21.06   |30.54   |26.36  |0.75  |
|6   |1496             |20.94   |30.37   |26.52  |0.75  |
|7   |1504             |20.87   |30.1    |26.7   |0.74  |
|8   |1500             |21.21   |30.65   |26.46  |0.76  |
|9   |1500             |20.84   |30.1    |26.27  |0.77  |
|10  |1500             |21.16   |30.64   |26.73  |0.76  |
|11  |1500             |21.32   |31.04   |26.53  |0.74  |
|12  |1500             |20.75   |30.12   |26.45

### Air Quality Dashboard Design

The Air Quality component of the SparkCity dashboard will focus on environmental
conditions that are useful to city planners without overstating small differences
in the synthetic sensor data.

The dashboard should include:

- **Current / Latest Conditions:** most recent available PM2.5, PM10, NO2, and CO
  averages, together with the timestamp and number of observations represented.
- **Daily Trend:** time-series visualization of average PM2.5 and PM10 measurements.
- **Monthly Environmental Trend:** monthly pollutant averages for longer-term planning
  and integration with convention-planning analysis.
- **Hourly Pattern:** average pollutant measurements by hour of day to identify whether
  meaningful time-of-day patterns exist.
- **Data Coverage:** observation and sensor counts so planners can distinguish complete
  periods from partial periods.
- **Drill-Down Capability:** filtering by date, sensor, and eventually SparkCity zone
  when zone enrichment is available.

The Air Quality data shows relatively small differences across months and hours.
Dashboard visualizations should therefore present the measured variation accurately
rather than labeling small differences as major environmental changes.

January 2026 represents a partial month and should be identified as such when compared
with complete monthly periods.

Air Quality is one input into the broader SparkCity planning analysis and should not
be used alone to determine overall convention suitability.

In [24]:
# Create latest available Air Quality conditions for dashboard summary cards

latest_timestamp = (
    air_quality_db_ready
    .agg(F.max("timestamp").alias("latest_timestamp"))
    .first()["latest_timestamp"]
)

latest_air_quality = (
    air_quality_db_ready
    .filter(F.col("timestamp") == latest_timestamp)
    .agg(
        F.count("*").alias("observation_count"),
        F.countDistinct("sensor_id").alias("sensor_count"),
        F.round(F.avg("pm25"), 2).alias("avg_pm25"),
        F.round(F.avg("pm10"), 2).alias("avg_pm10"),
        F.round(F.avg("no2"), 2).alias("avg_no2"),
        F.round(F.avg("co"), 2).alias("avg_co"),
        F.round(F.avg("humidity"), 2).alias("avg_humidity"),
    )
    .withColumn(
        "latest_timestamp",
        F.lit(latest_timestamp).cast("timestamp")
    )
    .select(
        "latest_timestamp",
        "observation_count",
        "sensor_count",
        "avg_pm25",
        "avg_pm10",
        "avg_no2",
        "avg_co",
        "avg_humidity",
    )
)

print("LATEST AVAILABLE AIR QUALITY CONDITIONS")
print("-" * 50)

latest_air_quality.show(truncate=False)

LATEST AVAILABLE AIR QUALITY CONDITIONS
--------------------------------------------------
+-------------------+-----------------+------------+--------+--------+-------+------+------------+
|latest_timestamp   |observation_count|sensor_count|avg_pm25|avg_pm10|avg_no2|avg_co|avg_humidity|
+-------------------+-----------------+------------+--------+--------+-------+------+------------+
|2026-01-11 04:45:00|1                |1           |31.11   |51.72   |32.33  |0.14  |58.24       |
+-------------------+-----------------+------------+--------+--------+-------+------+------------+



In [25]:
# Inspect coverage at the most recent Air Quality timestamps

recent_timestamp_coverage = (
    air_quality_db_ready
    .groupBy("timestamp")
    .agg(
        F.count("*").alias("observation_count"),
        F.countDistinct("sensor_id").alias("sensor_count"),
    )
    .orderBy(F.col("timestamp").desc())
)

print("RECENT AIR QUALITY TIMESTAMP COVERAGE")
print("-" * 50)

recent_timestamp_coverage.show(20, truncate=False)

RECENT AIR QUALITY TIMESTAMP COVERAGE
--------------------------------------------------
+-------------------+-----------------+------------+
|timestamp          |observation_count|sensor_count|
+-------------------+-----------------+------------+
|2026-01-11 04:45:00|1                |1           |
|2026-01-11 04:30:00|1                |1           |
|2026-01-11 04:15:00|1                |1           |
|2026-01-11 04:00:00|1                |1           |
|2026-01-11 03:45:00|1                |1           |
|2026-01-11 03:30:00|1                |1           |
|2026-01-11 03:15:00|1                |1           |
|2026-01-11 03:00:00|1                |1           |
|2026-01-11 02:45:00|1                |1           |
|2026-01-11 02:30:00|1                |1           |
|2026-01-11 02:15:00|1                |1           |
|2026-01-11 02:00:00|1                |1           |
|2026-01-11 01:45:00|1                |1           |
|2026-01-11 01:30:00|1                |1           |
|2026-01-1

## 8. Air Quality Monitoring & Alert Logic

The Air Quality pipeline includes monitoring logic to help identify measurements
that are unusually elevated relative to the historical SparkCity dataset.

These alerts are operational, data-driven indicators. They are not regulatory,
EPA, or public-health classifications.

Historical measurement distributions are used to establish monitoring thresholds.
This allows the dashboard to flag unusual conditions without presenting synthetic
SparkCity measurements as official health guidance.

In [26]:
# Calculate historical thresholds for Air Quality monitoring

air_quality_thresholds = (
    air_quality_db_ready
    .agg(
        F.expr("percentile_approx(pm25, 0.90)").alias("pm25_p90"),
        F.expr("percentile_approx(pm10, 0.90)").alias("pm10_p90"),
        F.expr("percentile_approx(no2, 0.90)").alias("no2_p90"),
        F.expr("percentile_approx(co, 0.90)").alias("co_p90"),
    )
    .first()
)

print("AIR QUALITY MONITORING THRESHOLDS")
print("-" * 45)
print(f"PM2.5 90th percentile: {air_quality_thresholds['pm25_p90']:.2f}")
print(f"PM10 90th percentile:  {air_quality_thresholds['pm10_p90']:.2f}")
print(f"NO2 90th percentile:   {air_quality_thresholds['no2_p90']:.2f}")
print(f"CO 90th percentile:    {air_quality_thresholds['co_p90']:.2f}")

AIR QUALITY MONITORING THRESHOLDS
---------------------------------------------
PM2.5 90th percentile: 32.14
PM10 90th percentile:  48.11
NO2 90th percentile:   41.29
CO 90th percentile:    1.27


In [27]:
# Create data-driven Air Quality monitoring alerts

pm25_threshold = air_quality_thresholds["pm25_p90"]
pm10_threshold = air_quality_thresholds["pm10_p90"]
no2_threshold = air_quality_thresholds["no2_p90"]
co_threshold = air_quality_thresholds["co_p90"]

air_quality_alerts = (
    air_quality_db_ready
    .withColumn(
        "pm25_alert",
        F.col("pm25") >= F.lit(pm25_threshold)
    )
    .withColumn(
        "pm10_alert",
        F.col("pm10") >= F.lit(pm10_threshold)
    )
    .withColumn(
        "no2_alert",
        F.col("no2") >= F.lit(no2_threshold)
    )
    .withColumn(
        "co_alert",
        F.col("co") >= F.lit(co_threshold)
    )
    .withColumn(
        "alert_count",
        F.col("pm25_alert").cast("int")
        + F.col("pm10_alert").cast("int")
        + F.col("no2_alert").cast("int")
        + F.col("co_alert").cast("int")
    )
    .withColumn(
        "alert_status",
        F.when(F.col("alert_count") > 0, "MONITOR")
        .otherwise("NORMAL")
    )
)

alert_summary = (
    air_quality_alerts
    .groupBy("alert_status")
    .count()
    .orderBy("alert_status")
)

print("AIR QUALITY MONITORING ALERT SUMMARY")
print("-" * 45)

alert_summary.show(truncate=False)

AIR QUALITY MONITORING ALERT SUMMARY
---------------------------------------------
+------------+-----+
|alert_status|count|
+------------+-----+
|MONITOR     |10814|
|NORMAL      |25186|
+------------+-----+



### Monitoring Alert Results

The Air Quality monitoring process evaluates PM2.5, PM10, NO2, and CO measurements
against their historical 90th-percentile values.

A record receives a `MONITOR` status when at least one pollutant meets or exceeds its
historical monitoring threshold. Otherwise, the record receives a `NORMAL` status.

Results:

- **NORMAL:** 25,186 observations
- **MONITOR:** 10,814 observations

Because each pollutant is evaluated independently, the percentage of observations
receiving a `MONITOR` status can exceed 10%. A measurement only needs to meet or
exceed one of the four pollutant thresholds to be flagged.

These statuses are intended to help city planners identify measurements that warrant
additional review. They are statistical monitoring indicators based on the SparkCity
dataset and should not be interpreted as EPA, regulatory, or public-health
classifications.

## 9. Pipeline Automation, Error Handling & Monitoring

The Air Quality workflow is designed to support automated execution as part of the
SparkCity data pipeline.

Each pipeline run should:

1. Load the Air Quality source data.
2. Validate required fields and primary-key uniqueness.
3. Apply the data-quality gate before persistence.
4. Compare incoming observations with existing PostgreSQL records.
5. Insert new records and update changed records when required.
6. Generate dashboard-ready Air Quality metrics.
7. Record pipeline execution status and processing statistics.
8. Stop safely and report an error when validation or database processing fails.

The pipeline is designed to be idempotent so repeated execution does not create
duplicate observations or unnecessarily update unchanged records.

Scheduling can be handled externally by a production scheduler such as cron,
Airflow, or another orchestration platform. This separates scheduling from the
Spark transformation logic and allows the same pipeline code to be executed
manually, on a schedule, or as part of a larger SparkCity workflow.

In [31]:
# Reusable Air Quality pipeline monitoring function

from datetime import datetime, timezone


def run_air_quality_pipeline_check(
    source_df,
    genuine_new_df,
    changed_df
):
    run_started = datetime.now(timezone.utc)

    pipeline_result = {
        "pipeline": "air_quality",
        "started_at_utc": run_started.isoformat(),
        "status": "RUNNING",
        "source_rows": 0,
        "invalid_rows": 0,
        "duplicate_keys": 0,
        "new_records": 0,
        "changed_records": 0,
        "database_write_required": False,
        "message": None,
    }

    try:
        pipeline_result["source_rows"] = source_df.count()

        invalid_condition = (
            F.col("sensor_id").isNull()
            | (F.trim(F.col("sensor_id")) == "")
            | F.col("timestamp").isNull()
            | F.col("location_lat").isNull()
            | F.col("location_lon").isNull()
            | F.col("pm25").isNull()
            | F.col("pm10").isNull()
            | F.col("no2").isNull()
            | F.col("co").isNull()
            | F.col("temperature").isNull()
            | F.col("humidity").isNull()
            | ~F.col("location_lat").between(-90, 90)
            | ~F.col("location_lon").between(-180, 180)
            | (F.col("pm25") < 0)
            | (F.col("pm10") < 0)
            | (F.col("no2") < 0)
            | (F.col("co") < 0)
            | ~F.col("humidity").between(0, 100)
        )

        pipeline_result["invalid_rows"] = (
            source_df
            .filter(invalid_condition)
            .count()
        )

        pipeline_result["duplicate_keys"] = (
            source_df
            .groupBy("sensor_id", "timestamp")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        if pipeline_result["invalid_rows"] > 0:
            raise ValueError(
                "Air Quality data-quality validation failed."
            )

        if pipeline_result["duplicate_keys"] > 0:
            raise ValueError(
                "Duplicate Air Quality primary keys detected."
            )

        pipeline_result["new_records"] = genuine_new_df.count()
        pipeline_result["changed_records"] = changed_df.count()

        pipeline_result["database_write_required"] = (
            pipeline_result["new_records"] > 0
            or pipeline_result["changed_records"] > 0
        )

        pipeline_result["status"] = "SUCCESS"

        if pipeline_result["database_write_required"]:
            pipeline_result["message"] = (
                "Validated database changes are available for persistence."
            )
        else:
            pipeline_result["message"] = (
                "Pipeline completed successfully; no database changes required."
            )

    except Exception as exc:
        pipeline_result["status"] = "FAILED"
        pipeline_result["message"] = str(exc)

    pipeline_result["completed_at_utc"] = (
        datetime.now(timezone.utc).isoformat()
    )

    return pipeline_result

In [32]:
# Execute the Air Quality pipeline validation and monitoring check

air_quality_pipeline_result = run_air_quality_pipeline_check(
    source_df=air_quality_db_ready,
    genuine_new_df=genuine_new_records,
    changed_df=changed_records,
)

print("AIR QUALITY PIPELINE RUN")
print("-" * 55)

for key, value in air_quality_pipeline_result.items():
    print(f"{key}: {value}")

AIR QUALITY PIPELINE RUN
-------------------------------------------------------
pipeline: air_quality
started_at_utc: 2026-09-16T02:17:55.063133+00:00
status: SUCCESS
source_rows: 36000
invalid_rows: 0
duplicate_keys: 0
new_records: 0
changed_records: 0
database_write_required: False
message: Pipeline completed successfully; no database changes required.
completed_at_utc: 2026-09-16T02:17:58.919057+00:00


## 10. Safe PostgreSQL Persistence & Upsert Strategy

The Air Quality pipeline must support future database changes without duplicating
observations or unnecessarily overwriting existing data.

The persistence workflow follows these rules:

1. Data must pass the Air Quality quality gate before persistence.
2. `(sensor_id, timestamp)` is used as the observation's unique business key.
3. New observations are eligible for INSERT processing.
4. Existing observations are eligible for UPDATE processing only when measurement
   values have changed.
5. Unchanged observations are not rewritten.
6. Timestamp normalization must be handled consistently before key comparison,
   particularly around daylight-saving-time transitions.
7. Database writes are skipped entirely when no validated changes are detected.
8. Persistence failures must be reported without treating a partially completed
   operation as a successful pipeline run.

The current Air Quality pipeline run identified zero genuine new observations and
zero changed observations. Therefore, no PostgreSQL write is required for this run.

The existing `sparkcity.air_quality` source table will not be overwritten. Future
writes should use a controlled PostgreSQL upsert process so the table's primary-key
and validation constraints remain enforced.

In [33]:
# PostgreSQL client used for controlled Air Quality upserts

import psycopg

print(f"Psycopg version: {psycopg.__version__}")
print("PostgreSQL upsert support ready.")

Psycopg version: 3.3.5
PostgreSQL upsert support ready.


## 11. Day 5 Summary & Deployment Documentation

### Air Quality Pipeline Completion

Day 5 completed the database integration and production-readiness work for the
SparkCity Air Quality dataset.

The completed Air Quality workflow includes:

- Apache Spark integration with the shared SparkCity PostgreSQL database
- secure database configuration using environment variables
- validation of 36,000 Air Quality observations before persistence
- primary-key and duplicate validation using `(sensor_id, timestamp)`
- validation of coordinates, pollutant measurements, humidity, and required fields
- comparison of processed Air Quality data with the existing PostgreSQL source table
- identification and documentation of daylight-saving-time timestamp ambiguity
- verification that 35,996 directly matching PostgreSQL records contain no changed
  measurement values
- identification of zero genuine new observations and zero required updates
- idempotent pipeline behavior that prevents unnecessary database writes
- Air Quality database/star-schema design documentation
- daily, monthly, and hourly dashboard-ready Air Quality aggregations
- data-driven Air Quality monitoring indicators
- reusable pipeline validation, error handling, and execution-status reporting

### Database Persistence Result

The current pipeline run completed successfully with:

- **Source records:** 36,000
- **Invalid records:** 0
- **Duplicate primary keys:** 0
- **New records requiring persistence:** 0
- **Existing records requiring updates:** 0
- **Database write required:** No

Because the PostgreSQL Air Quality source is already synchronized with the validated
dataset, the pipeline correctly skips unnecessary database writes.

The persistence design supports future insert/update processing when validated changes
are detected while preserving the existing `sparkcity.air_quality` source table and
its database constraints.

### Monitoring

Air Quality monitoring uses historical pollutant distributions to identify
measurements that warrant additional review.

The monitoring classifications are statistical indicators derived from the SparkCity
dataset. They are not EPA, regulatory, or public-health classifications.

Pipeline execution also records validation results, database-change requirements,
success/failure status, and processing timestamps so failures can be identified before
data is persisted.

### Automation & Recovery

The Air Quality workflow is structured so it can be executed manually or scheduled
as part of the larger SparkCity pipeline.

A production deployment can use an external scheduler or orchestration platform to
execute the pipeline on a defined schedule. Keeping scheduling separate from the Spark
processing logic allows the same workflow to support development, testing, and
production execution.

If validation fails, the pipeline reports a failed status rather than proceeding with
persistence. If no new or changed records are detected, database processing is safely
skipped.

### Dashboard Integration

Air Quality dashboard-ready datasets were created for:

- daily pollutant trends
- monthly environmental trends
- hourly pollutant patterns
- observation and sensor coverage
- monitoring indicators

These outputs are designed to be integrated with the other SparkCity datasets in the
team's final city-operations dashboard rather than deployed as a separate Air Quality
dashboard.

Air Quality represents one environmental input into the broader SparkCity planning
analysis and should be considered alongside traffic, weather, energy, occupancy,
fiscal, and city-zone information.

### Day 5 Status

**Air Quality Day 5 implementation is complete and ready for integration with the
team's final SparkCity dashboard and deployment workflow.**